# Show-o2: shared-image gradient analysis

40 images · 4 categories · 10 examples per category · 28 layers · 49 flow times

Each example pairs understanding and generation losses on the same image. U predicts an explicitly formatted answer; G uses a descriptive caption and that image as its flow-matching target. Show-o2 weights are never updated.

Results: mean ± sample standard deviation across examples, by category and overall. Steps and layers are repeated measurements, not extra examples. This is a new diagnostic, not the previous MLP test split.

## 1. Install and clone
Fresh Colab GPU runtime; A100 and high host RAM recommended. Runtime and peak memory are not yet measured for this version.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.46.3", "tokenizers==0.20.3", "diffusers==0.30.1",
    "accelerate", "einops", "omegaconf", "timm", "decord", "torchdiffeq",
    "sentencepiece", "Pillow", "numpy", "pandas", "matplotlib",
], check=True)

REPO_ROOT = Path("/content/FlashU-Lab")
REPO_BRANCH = "feature/pilot-acceleration-experiment"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
    "https://github.com/Nicerova7/FlashU-Lab.git", str(REPO_ROOT),
], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))


## 2. Imports and paths
Results go to a new Drive folder. Small input records are cached on Colab disk.


In [ ]:
import gc
import json
import textwrap
from datetime import datetime, timezone
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display
from omegaconf import OmegaConf

from models import Showo2Qwen2_5, WanVAE, omni_attn_mask_naive
from models.misc import get_text_tokenizer, interpolate_pos_encoding
from utils import denorm, path_to_llm_name
from evaluation.calculate_token_importance_understanding import assemble_prefill
from evaluation.calculate_token_importance_generation import prepare_generation_inputs
from evaluation.calculate_gradient_update_generation import measured_backbone

drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/intership/acceleration-project")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUTPUT_DIR = DATA_ROOT / "component_gradient_cosine" / f"shared_image_40_{RUN_ID}"
CACHE_DIR = Path("/content/shared_image_inputs") / RUN_ID
OUTPUT_DIR.mkdir(parents=True)
CACHE_DIR.mkdir(parents=True)

DEVICE = torch.device("cuda:0")
DTYPE = torch.bfloat16
BASE_SEED = 42
FLOW_TIMES = torch.linspace(0, 1, 50)[:-1].tolist()
CATEGORIES = [
    "counting", "spatial_relations",
    "attribute_combinations", "visual_state_traffic_lights",
]
CATEGORY_LABELS = {
    "counting": "Counting",
    "spatial_relations": "Spatial relations",
    "attribute_combinations": "Vehicle attributes",
    "visual_state_traffic_lights": "Traffic-light color",
    "all": "All 40 examples",
}
COLORS = {"attention": "#0072B2", "ffn": "#D55E00"}

plt.rcParams.update({
    "font.size": 15, "axes.titlesize": 17, "axes.labelsize": 15,
    "xtick.labelsize": 13, "ytick.labelsize": 13,
    "legend.fontsize": 13, "figure.dpi": 120, "savefig.dpi": 200,
})
pd.set_option("display.max_colwidth", 100)

def show_table(frame, precision=4):
    display(
        frame.style.format(precision=precision)
        .hide(axis="index")
        .set_properties(**{"font-size": "14px", "padding": "8px"})
        .set_table_styles([
            {"selector": "th", "props": [("font-size", "14px"), ("padding", "8px")]},
        ])
    )


## 3. Build 40 shared-image examples
Fixed IDs 000000–000009 per category. Captions follow the supplied annotations; no autogen references or MLP splits are used. The traffic-light subset contains only red labels.


In [ ]:
COUNTING_CAPTIONS = [
    "a photo of one dog", "a photo of one bear",
    "a photo of one elephant", "a photo of one train",
    "a photo of one cat", "a photo of two zebras",
    "a photo of two elephants", "a photo of two cows",
    "a photo of two giraffes", "a photo of two bears",
]
SPATIAL_CAPTIONS = [
    "a photo of a dog left of a fire hydrant",
    "a photo of a motorcycle left of a person",
    "a photo of a person left of a cat",
    "a photo of a bicycle left of a person",
    "a photo of a dog left of a cat",
    "a photo of a person right of a motorcycle",
    "a photo of a cat right of a bottle",
    "a photo of a dog right of a bench",
    "a photo of a cup right of a bottle",
    "a photo of a car right of a motorcycle",
]
CAPTIONS = {
    "counting": COUNTING_CAPTIONS,
    "spatial_relations": SPATIAL_CAPTIONS,
    "attribute_combinations": ["a photo of a red car"] * 5 + ["a photo of a blue car"] * 5,
    "visual_state_traffic_lights": ["a photo of a traffic light showing red"] * 10,
}
ANSWER_OPTIONS = {
    "counting": "One, Two, Three, Four",
    "spatial_relations": "Left, Right",
    "attribute_combinations": "Red car, Blue car, Red bus, Blue bus",
    "visual_state_traffic_lights": "Red, Yellow, Green",
}

examples = []
for category in CATEGORIES:
    for index in range(10):
        example_id = len(examples) + 1
        source_path = (
            DATA_ROOT / "gradient_update_understanding"
            / category / f"sample_{index:06d}.pt"
        )
        print(f"Loading {example_id:02d}/40: {category}/{index:06d}", flush=True)
        saved = torch.load(source_path, map_location="cpu", weights_only=True, mmap=True)
        source = saved["sample"]["source_record"]
        answer = str(source["expected_answer"]).strip().capitalize()
        question = source["prompt"] + (
            f"\nAnswer with exactly one of: {ANSWER_OPTIONS[category]}. "
            "Use the capitalization shown. Do not add punctuation or explanation."
        )
        record = dict(
            example_id=example_id, category=category,
            uid=f"U/{category}/sample_{index:06d}",
            source_path=str(source_path), source_run_id=saved["run_id"],
            source_image_path=source["image_path"],
            image_sha256=saved["sample"]["image_sha256"],
            original_question=source["prompt"], original_answer=source["expected_answer"],
            u_prompt=question, u_answer=answer, g_prompt=CAPTIONS[category][index],
            noise_seed=BASE_SEED + example_id - 1,
            cache_path=str(CACHE_DIR / f"example_{example_id:02d}.pt"),
        )
        torch.save(
            {**record, "image_latents": saved["image_latents"].clone()},
            record["cache_path"],
        )
        examples.append(record)
        del saved

manifest = pd.DataFrame(examples)
manifest.to_csv(OUTPUT_DIR / "examples.csv", index=False)
show_table(
    manifest.groupby("category", sort=False).size()
    .rename("examples").reset_index()
)
for category in CATEGORIES:
    print(CATEGORY_LABELS[category])
    show_table(
        manifest.loc[manifest["category"] == category,
                     ["example_id", "original_question", "u_answer", "g_prompt"]]
    )


## 4. Load Show-o2
Differentiate Q/K/V/O and gate/up/down parameters, including biases. No optimizer or acceleration patches.


In [ ]:
config = OmegaConf.load(REPO_ROOT / "configs/showo2_1.5b_demo_432x432.yaml")
llm_path = config.model.showo.llm_model_path
tokenizer, token_ids = get_text_tokenizer(
    llm_path, add_showo_tokens=True, return_showo_token_ids=True,
    llm_name=path_to_llm_name[llm_path],
)
model = Showo2Qwen2_5.from_pretrained(
    config.model.showo.pretrained_model_path, use_safetensors=False,
).to(device=DEVICE, dtype=DTYPE).eval().requires_grad_(False)
core = model.showo.model
core.config.use_cache = False
resources = SimpleNamespace(
    model=model, tokenizer=tokenizer, token_ids=token_ids,
    device=DEVICE, dtype=DTYPE,
)

groups = []
for layer_id, layer in enumerate(core.layers):
    for component, module_name, projections in [
        ("attention", "self_attn", ("q_proj", "k_proj", "v_proj", "o_proj")),
        ("ffn", "mlp", ("gate_proj", "up_proj", "down_proj")),
    ]:
        module = getattr(layer, module_name)
        named = [
            (f"showo.model.layers.{layer_id}.{module_name}.{projection}.{name}", parameter)
            for projection in projections
            for name, parameter in getattr(module, projection).named_parameters()
        ]
        for name, parameter in named:
            parameter.requires_grad_(True)
        groups.append(dict(
            layer=layer_id, component=component,
            names=[name for name, parameter in named],
            parameters=[parameter for name, parameter in named],
        ))

parameters = [parameter for group in groups for parameter in group["parameters"]]
parameter_groups = [{
    "layer": group["layer"], "component": group["component"],
    "names": group["names"],
    "shapes": [list(parameter.shape) for parameter in group["parameters"]],
    "num_parameters": sum(parameter.numel() for parameter in group["parameters"]),
} for group in groups]
(OUTPUT_DIR / "parameter_groups.json").write_text(
    json.dumps(parameter_groups, indent=2), encoding="utf-8",
)
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Decoder layers:", len(core.layers))
print("Selected parameters:", sum(parameter.numel() for parameter in parameters))


## 5. Losses and cosine
U: answer-token mean cross-entropy. G: conditional velocity MSE on x(t) = (1−t) noise + t image, targeting image − noise. These are linear interpolation states, not the sampled generation trajectory. Matrix-wise sums give the concatenated-vector cosine; zero norms yield NaN.


In [ ]:
def understanding_objective(sample):
    answer_ids = tokenizer(
        sample["u_answer"], add_special_tokens=False,
    )["input_ids"]
    embeds, _, _, layout = assemble_prefill(
        model, tokenizer, token_ids, sample["image_latents"],
        sample["u_prompt"],
        int(config.dataset.preprocessing.max_seq_length) - len(answer_ids),
        omni_attn_mask_naive, interpolate_pos_encoding,
    )
    length = embeds.shape[1]
    answer = torch.tensor([answer_ids], device=DEVICE)
    embeds = torch.cat([embeds, core.embed_tokens(answer)], dim=1).detach()
    offset, span = layout["attention_span"]
    modalities = torch.tensor([[[offset, span]]], device=DEVICE)
    mask = omni_attn_mask_naive(
        B=1, LEN=embeds.shape[1], modalities=modalities,
        device=DEVICE, inverted=True,
    ).to(DTYPE)
    hidden = core(
        inputs_embeds=embeds, attention_mask=mask,
        use_cache=False, output_hidden_states=False, return_dict=True,
    ).last_hidden_state
    logits = model.showo.lm_head(hidden[:, length - 1:-1]).float()
    loss = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]), answer.reshape(-1),
    )
    log_probs = logits.detach().log_softmax(dim=-1)[0]
    token_rows = []
    for position, target_id in enumerate(answer_ids):
        top_id = log_probs[position].argmax().item()
        token_rows.append(dict(
            position=position, target_id=target_id,
            target_token=tokenizer.decode([target_id]),
            target_probability=log_probs[position, target_id].exp().item(),
            token_loss=-log_probs[position, target_id].item(),
            top_token=tokenizer.decode([top_id]),
            top_probability=log_probs[position, top_id].exp().item(),
        ))
    return loss, token_rows


def generation_backbone(*, inputs_embeds, attention_mask, **kwargs):
    hidden = core(
        inputs_embeds=inputs_embeds, attention_mask=attention_mask,
        use_cache=False, output_hidden_states=False, return_dict=True,
    ).last_hidden_state
    return {"logits": None, "hidden_states": (hidden,)}


def generation_inputs(sample):
    return prepare_generation_inputs(
        {"prompt": sample["g_prompt"]}, resources, config,
        {"guidance_scale": 0.0},
    )


def sample_noise(sample):
    generator = torch.Generator(device=DEVICE).manual_seed(sample["noise_seed"])
    return torch.randn(
        sample["image_latents"].shape, generator=generator,
        device=DEVICE, dtype=DTYPE,
    )


def generation_objective(inputs, noise, reference, timestep):
    latent = (1.0 - timestep) * noise + timestep * reference
    time = torch.tensor([timestep], device=DEVICE, dtype=latent.dtype)
    with measured_backbone(model, SimpleNamespace(forward=generation_backbone)):
        _, velocity = model(
            text_tokens=inputs["text_tokens"], image_latents=latent, t=time,
            attention_mask=inputs["attention_mask"],
            modality_positions=inputs["modality_positions"],
            max_seq_len=inputs["sequence_length"],
            guidance_scale=0.0, device=str(DEVICE),
        )
    return F.mse_loss(velocity.float(), (reference - noise).float())


@torch.no_grad()
def component_cosines(first_gradients, second_gradients):
    results = []
    start = 0
    for group in groups:
        end = start + len(group["parameters"])
        dot = torch.zeros((), device=DEVICE)
        first_squared = torch.zeros((), device=DEVICE)
        second_squared = torch.zeros((), device=DEVICE)
        for first_gradient, second_gradient in zip(
            first_gradients[start:end], second_gradients[start:end],
        ):
            first = first_gradient.to(device=DEVICE, dtype=torch.float32)
            second = second_gradient.detach().float()
            dot += (first * second).sum()
            first_squared += first.square().sum()
            second_squared += second.square().sum()
        results.append(dict(
            layer=group["layer"], component=group["component"],
            cosine=(dot / (first_squared.sqrt() * second_squared.sqrt())).item(),
            first_norm=first_squared.sqrt().item(),
            second_norm=second_squared.sqrt().item(),
        ))
        start = end
    return results


## 6. Measure 10 examples per category
40 U and 1,960 G backward passes; no cross-product of examples. Save each example before continuing. A new run folder keeps previous experiments intact.


In [ ]:
run = dict(
    run_id=RUN_ID, protocol="shared_image_40_v1",
    repo_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    repo_branch=REPO_BRANCH,
    checkpoint=config.model.showo.pretrained_model_path,
    model_config=dict(model.config), backbone_config=core.config.to_dict(),
    config=OmegaConf.to_container(config, resolve=True),
    torch_version=torch.__version__, dtype=str(DTYPE),
    gpu=torch.cuda.get_device_name(DEVICE),
    num_examples=len(examples), categories=CATEGORIES,
    pairing="one U/G objective pair per saved U image",
    reference_source="saved understanding image latent; no autogen references",
    understanding_loss="mean cross entropy over explicitly formatted answer tokens",
    generation_loss="conditional linear-path velocity MSE; target image minus noise",
    flow_times=FLOW_TIMES, base_seed=BASE_SEED,
    noise_policy="one BF16 noise draw per image, seed = base_seed + example_id - 1",
    captions="annotation-based concept captions, not full-scene descriptions",
    model_weights_updated=False,
)
(OUTPUT_DIR / "run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
token_rows = []
loss_rows = []

for example in examples:
    sample = torch.load(example["cache_path"], map_location="cpu", weights_only=True)
    print(
        f"Example {sample['example_id']:02d}/40 | {sample['category']} | "
        f"{sample['u_answer']} | {sample['g_prompt']}",
        flush=True,
    )
    u_loss, tokens = understanding_objective(sample)
    u_value = u_loss.item()
    u_gradients = tuple(
        gradient.detach().cpu()
        for gradient in torch.autograd.grad(u_loss, parameters)
    )
    token_rows.extend([
        dict(example_id=sample["example_id"], category=sample["category"], **token)
        for token in tokens
    ])
    loss_rows.append(dict(
        example_id=sample["example_id"], category=sample["category"],
        task="U", step=-1, timestep=1.0, loss=u_value,
    ))
    del u_loss

    inputs = generation_inputs(sample)
    noise = sample_noise(sample)
    reference = sample["image_latents"].to(device=DEVICE, dtype=DTYPE)
    example_rows = []

    for step, timestep in enumerate(FLOW_TIMES):
        g_loss = generation_objective(inputs, noise, reference, timestep)
        g_value = g_loss.item()
        g_gradients = torch.autograd.grad(g_loss, parameters)
        for result in component_cosines(u_gradients, g_gradients):
            example_rows.append(dict(
                example_id=sample["example_id"], category=sample["category"],
                uid=sample["uid"], step=step, timestep=timestep,
                u_loss=u_value, g_loss=g_value,
                u_norm=result["first_norm"], g_norm=result["second_norm"],
                layer=result["layer"], component=result["component"],
                cosine=result["cosine"],
            ))
        loss_rows.append(dict(
            example_id=sample["example_id"], category=sample["category"],
            task="G", step=step, timestep=timestep, loss=g_value,
        ))
        print(
            f"  Step {step:02d}/48 | t={timestep:.3f} | "
            f"U loss={u_value:.4f} | G loss={g_value:.4f}",
            flush=True,
        )
        del g_loss, g_gradients

    pd.DataFrame(example_rows).to_csv(
        OUTPUT_DIR / f"cosine_example_{sample['example_id']:02d}.csv", index=False,
    )
    pd.DataFrame(token_rows).to_csv(OUTPUT_DIR / "answer_tokens.csv", index=False)
    pd.DataFrame(loss_rows).to_csv(OUTPUT_DIR / "losses.csv", index=False)
    print(f"Saved example {sample['example_id']:02d}/40", flush=True)
    del sample, inputs, noise, reference, u_gradients, example_rows
    gc.collect()
    torch.cuda.empty_cache()

print("Measurement finished:", OUTPUT_DIR)


## 7. Summarize across examples
Layer view: average times within each example, then mean ± std across examples. Time view: average layers within each example, then mean ± std across examples. Report n=10 per category and n=40 overall; valid counts accompany averages.


In [ ]:
details = pd.concat([
    pd.read_csv(OUTPUT_DIR / f"cosine_example_{example['example_id']:02d}.csv")
    for example in examples
], ignore_index=True)
details.to_csv(OUTPUT_DIR / "cosine_per_example_step.csv", index=False)

per_layer = (
    details.groupby(["example_id", "category", "layer", "component"], as_index=False)
    .agg(cosine=("cosine", "mean"), valid_times=("cosine", "count"),
         total_times=("step", "size"))
)
per_time = (
    details.groupby(
        ["example_id", "category", "step", "timestep", "component"], as_index=False,
    )
    .agg(cosine=("cosine", "mean"), valid_layers=("cosine", "count"),
         total_layers=("layer", "size"))
)
layer_pool = pd.concat([per_layer, per_layer.assign(category="all")], ignore_index=True)
time_pool = pd.concat([per_time, per_time.assign(category="all")], ignore_index=True)
layer_summary = (
    layer_pool.groupby(["category", "layer", "component"], as_index=False)
    .agg(mean=("cosine", "mean"), std=("cosine", "std"), n=("cosine", "count"))
)
time_summary = (
    time_pool.groupby(["category", "step", "timestep", "component"], as_index=False)
    .agg(mean=("cosine", "mean"), std=("cosine", "std"), n=("cosine", "count"))
)

per_layer.to_csv(OUTPUT_DIR / "cosine_per_example_layer.csv", index=False)
per_time.to_csv(OUTPUT_DIR / "cosine_per_example_time.csv", index=False)
layer_summary.to_csv(OUTPUT_DIR / "layer_mean_std.csv", index=False)
time_summary.to_csv(OUTPUT_DIR / "time_mean_std.csv", index=False)

raw_pool = pd.concat([details, details.assign(category="all")], ignore_index=True)
distribution = (
    raw_pool.groupby(["category", "component"], as_index=False)
    .agg(
        signed_mean=("cosine", "mean"),
        mean_absolute=("cosine", lambda values: values.abs().mean()),
        p05=("cosine", lambda values: values.quantile(0.05)),
        p95=("cosine", lambda values: values.quantile(0.95)),
        negative_fraction=("cosine", lambda values: (values.dropna() < 0).mean()),
        valid_measurements=("cosine", "count"),
        total_measurements=("cosine", "size"),
    )
)
distribution.to_csv(OUTPUT_DIR / "unaveraged_distribution.csv", index=False)
show_table(distribution)

plot_limit = 1.15 * max(
    (layer_summary["mean"].abs() + layer_summary["std"]).max(),
    (time_summary["mean"].abs() + time_summary["std"]).max(),
)

def plot_results(category):
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
    for ax, summary, x, title, xlabel in [
        (axes[0], layer_summary, "layer",
         "By layer · flow times averaged", "Decoder layer"),
        (axes[1], time_summary, "timestep",
         "By flow time · layers averaged", "Flow time: noise → image"),
    ]:
        means = summary[summary["category"] == category]
        for component, color in COLORS.items():
            curve = means[means["component"] == component].sort_values(x)
            ax.plot(curve[x], curve["mean"], color=color, linewidth=2.5, label=component)
            ax.fill_between(
                curve[x].to_numpy(),
                (curve["mean"] - curve["std"]).to_numpy(),
                (curve["mean"] + curve["std"]).to_numpy(),
                color=color, alpha=0.18,
            )
        ax.axhline(0, color="black", linestyle="--", linewidth=1)
        ax.set(title=title, xlabel=xlabel, ylim=(-plot_limit, plot_limit))
        ax.legend(frameon=False)
        ax.grid(axis="y", alpha=0.2)

    count = layer_summary.loc[layer_summary["category"] == category, "n"].max()
    axes[0].set_ylabel("U–G gradient cosine · zoomed scale")
    fig.suptitle(
        f"{CATEGORY_LABELS[category]} · {count} examples · mean ± std",
        fontsize=20,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{category}_cosine.png", bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{category}_cosine.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 8. Results by category
Ten examples per category. All panels share the same zoomed y-axis; shading is standard deviation, not a confidence interval.


In [ ]:
for category in CATEGORIES:
    plot_results(category)
    selected = layer_summary[layer_summary["category"] == category].copy()
    selected["mean ± std"] = [
        f"{mean:+.4f} ± {std:.4f}"
        for mean, std in zip(selected["mean"], selected["std"])
    ]
    compact = selected.pivot(
        index="layer", columns="component", values="mean ± std",
    ).reset_index()
    print(CATEGORY_LABELS[category])
    show_table(compact)


## 9. Overall results
Equal category counts give equal category weight. These selected examples are a diagnostic set, not an independent benchmark.


In [ ]:
plot_results("all")

per_example_mean = (
    per_layer.groupby(["example_id", "category", "component"], as_index=False)
    .agg(cosine=("cosine", "mean"))
)
overview_pool = pd.concat([
    per_example_mean, per_example_mean.assign(category="all"),
], ignore_index=True)
overview = (
    overview_pool.groupby(["category", "component"], as_index=False)
    .agg(mean=("cosine", "mean"), std=("cosine", "std"), n=("cosine", "count"))
)
overview["mean ± std"] = [
    f"{mean:+.4f} ± {std:.4f}"
    for mean, std in zip(overview["mean"], overview["std"])
]
overview.to_csv(OUTPUT_DIR / "overview_mean_std.csv", index=False)
show_table(overview[["category", "component", "mean ± std", "n"]])


## 10. Repeatability controls
Repeat U twice and G twice at time index 24 for the first example of each category. Same-objective cosines should be near +1. This checks repeatability, not the validity of the hypothesis.


In [ ]:
representatives = [examples[index] for index in [0, 10, 20, 30]]
control_rows = []

for example in representatives:
    sample = torch.load(example["cache_path"], map_location="cpu", weights_only=True)
    print("Checking:", sample["category"], flush=True)

    first_loss, _ = understanding_objective(sample)
    first_value = first_loss.item()
    first_gradients = tuple(
        gradient.detach().cpu()
        for gradient in torch.autograd.grad(first_loss, parameters)
    )
    del first_loss
    second_loss, _ = understanding_objective(sample)
    second_value = second_loss.item()
    second_gradients = torch.autograd.grad(second_loss, parameters)
    for result in component_cosines(first_gradients, second_gradients):
        control_rows.append(dict(
            category=sample["category"], task="U vs U",
            first_loss=first_value, second_loss=second_value, **result,
        ))
    del first_gradients, second_gradients, second_loss

    inputs = generation_inputs(sample)
    noise = sample_noise(sample)
    reference = sample["image_latents"].to(device=DEVICE, dtype=DTYPE)
    first_loss = generation_objective(inputs, noise, reference, FLOW_TIMES[24])
    first_value = first_loss.item()
    first_gradients = tuple(
        gradient.detach().cpu()
        for gradient in torch.autograd.grad(first_loss, parameters)
    )
    del first_loss
    second_loss = generation_objective(inputs, noise, reference, FLOW_TIMES[24])
    second_value = second_loss.item()
    second_gradients = torch.autograd.grad(second_loss, parameters)
    for result in component_cosines(first_gradients, second_gradients):
        control_rows.append(dict(
            category=sample["category"], task="G vs G",
            first_loss=first_value, second_loss=second_value, **result,
        ))
    del first_gradients, second_gradients, second_loss
    del sample, inputs, noise, reference
    gc.collect()
    torch.cuda.empty_cache()

controls = pd.DataFrame(control_rows)
controls.to_csv(OUTPUT_DIR / "repeatability_per_layer.csv", index=False)
control_summary = (
    controls.groupby(["category", "task", "component"], as_index=False)
    .agg(
        min_cosine=("cosine", "min"), mean_cosine=("cosine", "mean"),
        first_loss=("first_loss", "first"), second_loss=("second_loss", "first"),
    )
)
control_summary["loss_difference"] = (
    control_summary["first_loss"] - control_summary["second_loss"]
).abs()
control_summary.to_csv(OUTPUT_DIR / "repeatability_summary.csv", index=False)
show_table(control_summary)


## 11. One-minute review
Four fixed examples: shared input image, U question/answer, G caption, and raw U–G cosines at time index 24. Controls and annotations support an audit; they do not prove a universal conflict pattern.


In [ ]:
vae = WanVAE(
    vae_pth=str(DATA_ROOT / "model_weights" / "Wan2.1_VAE.pth"),
    dtype=DTYPE, device=DEVICE,
)
fig, axes = plt.subplots(2, 2, figsize=(18, 17))

for ax, example in zip(axes.flat, representatives):
    sample = torch.load(example["cache_path"], map_location="cpu", weights_only=True)
    with torch.no_grad():
        pixels = vae.batch_decode(
            sample["image_latents"].to(DEVICE).unsqueeze(2)
        ).squeeze(2)
        image = denorm(pixels)[0]
    ax.imshow(image)
    ax.set_title(CATEGORY_LABELS[sample["category"]])
    ax.set_xticks([])
    ax.set_yticks([])
    caption = (
        f"U: {sample['original_question']}\n"
        f"Target: {sample['u_answer']}\n"
        f"G: {sample['g_prompt']}"
    )
    ax.set_xlabel(
        "\n".join(textwrap.fill(line, width=65) for line in caption.splitlines()),
        fontsize=14,
    )
    del sample, pixels

fig.suptitle("The same saved image is used for U and G", fontsize=22)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "review_shared_images.png", bbox_inches="tight")
plt.show()
plt.close(fig)
del vae
gc.collect()
torch.cuda.empty_cache()

fig, axes = plt.subplots(2, 2, figsize=(18, 11), sharey=True)
single_time = details[details["step"] == 24]
for ax, example in zip(axes.flat, representatives):
    selected = single_time[single_time["example_id"] == example["example_id"]]
    for component, color in COLORS.items():
        curve = selected[selected["component"] == component]
        ax.plot(
            curve["layer"], curve["cosine"], color=color,
            marker="o", linewidth=2, label=component,
        )
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set(
        title=CATEGORY_LABELS[example["category"]],
        xlabel="Decoder layer", ylabel="U–G gradient cosine",
    )
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.2)

fig.suptitle("Four fixed examples · flow time index 24 · no time averaging", fontsize=20)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "review_single_time.png", bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "review_single_time.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

answer_tokens = pd.read_csv(OUTPUT_DIR / "answer_tokens.csv")
representative_ids = [example["example_id"] for example in representatives]
print("Expected answer tokens and model predictions")
show_table(answer_tokens[answer_tokens["example_id"].isin(representative_ids)])

print("Repeatability: same-objective cosines should be near +1")
show_table(control_summary[["category", "task", "component", "min_cosine", "loss_difference"]])

print("Unaveraged measurements: distinguish small values from sign cancellation")
show_table(distribution[distribution["category"] == "all"])

norm_audit = (
    details.assign(
        nonfinite_cosine=~np.isfinite(details["cosine"]),
        zero_u_norm=details["u_norm"] == 0,
        zero_g_norm=details["g_norm"] == 0,
    )
    .groupby("component", as_index=False)
    .agg(
        nonfinite_cosines=("nonfinite_cosine", "sum"),
        zero_u_norms=("zero_u_norm", "sum"),
        zero_g_norms=("zero_g_norm", "sum"),
        min_u_norm=("u_norm", "min"),
        min_g_norm=("g_norm", "min"),
    )
)
norm_audit.to_csv(OUTPUT_DIR / "norm_audit.csv", index=False)
print("Numerical audit")
show_table(norm_audit)

endpoints = (
    per_layer.pivot(
        index=["example_id", "category", "component"],
        columns="layer", values="cosine",
    )[[0, 27]]
    .rename(columns={0: "first", 27: "last"})
    .reset_index()
)
endpoints["change"] = endpoints["last"] - endpoints["first"]
endpoint_pool = pd.concat([
    endpoints, endpoints.assign(category="all"),
], ignore_index=True)
endpoint_summary = (
    endpoint_pool.groupby(["category", "component"], as_index=False)
    .agg(
        first_mean=("first", "mean"), first_std=("first", "std"),
        last_mean=("last", "mean"), last_std=("last", "std"),
        change_mean=("change", "mean"), change_std=("change", "std"),
        n=("change", "count"),
    )
)
endpoint_summary.to_csv(OUTPUT_DIR / "first_last_layer_summary.csv", index=False)
print("First and last layers: flow times averaged within each example")
show_table(endpoint_summary)

print("Positive: aligned gradients. Negative: opposing gradients. Near zero: weak directional alignment.")
print("No conclusion about depth or components is assumed.")
print("Results:", OUTPUT_DIR)
